# LLM Engineering: Landscape & Interview Map

LLM-specific MLE interviews have emerged as a distinct track at AI-first companies. This note maps the topics covered, the key decisions interviewers probe, and the glossary you need to navigate LLM system design questions confidently.

## What Interviewers Test
- When to fine-tune vs prompt engineer vs use RAG — the decision framework
- Open vs closed model tradeoffs for production systems
- Scaling law intuitions and their implications for model selection
- Transformer architecture decisions that affect inference (GQA, RoPE, KV cache)
- Enough glossary fluency to discuss any of the above in 5 minutes

## Decoder-Only vs Encoder-Decoder

| | Decoder-Only (GPT, LLaMA) | Encoder-Decoder (T5, BART) |
|---|---|---|
| **Primary use** | Generation, completion, chat | Seq2seq: translation, summarization |
| **Self-attention** | Causal (can't see future tokens) | Full (encoder) + causal (decoder) |
| **KV cache** | Efficient (fixed past) | Complex (two caches) |
| **In-context learning** | Excellent | Weaker |
| **Fine-tuning** | Standard for chat/instruction | Standard for structured output |
| **Training data** | Autoregressive (next-token prediction) | Encoder-decoder cross-entropy |

Modern trend: decoder-only dominates (GPT-4, Claude, Gemini, LLaMA) because it unifies all tasks under generation and in-context learning excels.


## The Build vs Buy vs Fine-Tune Decision Framework

```
Is the task solvable with a closed API + prompting?
  YES → Use API (fastest, lowest cost to start)
  NO  ↓
Is data privacy / latency / cost a hard constraint?
  YES → Open model (self-hosted)
  NO  → API with fine-tuning endpoint
  ↓ (either path)
Does the task require knowledge not in pretrained model?
  YES → RAG (if knowledge is retrievable) or Fine-tuning (if it's a skill)
  NO  → Prompting + few-shot
```


## Open vs Closed Model Tradeoff

| Dimension | Closed (GPT-4, Claude) | Open (LLaMA, Mistral) |
|---|---|---|
| **Quality** | Best in class (at release) | Competitive for many tasks |
| **Cost** | Per-token pricing | Infra cost; large upfront |
| **Latency** | Network round-trip | Controllable (local) |
| **Privacy** | Data sent to third party | Data stays on-prem |
| **Customization** | Limited (fine-tune API) | Full control |
| **Maintenance** | Zero | Model updates, serving infra |
| **Best for** | Prototyping, varied tasks, low volume | Scale, privacy, specialized domains |


## Scaling Laws Intuition

Chinchilla scaling laws (Hoffmann et al., 2022): Optimal compute allocation gives:
- **Model size N** and **tokens T** should scale proportionally: $T \approx 20N$
- A 7B model should be trained on ~140B tokens for "compute-optimal" training
- In practice, inference cost matters more than training cost, so models are often "over-trained"

**Implications for practitioners:**
- Bigger models aren't always better for a fixed inference budget — a smaller model trained longer can match a larger undertrained model
- For fine-tuning: the pretrained model's general capability is the ceiling; fine-tuning specializes within it


In [ ]:
# Scaling law illustration (approximate relationships)
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Chinchilla-style: loss ~ C^(-alpha) for compute budget C
params = np.array([1e9, 7e9, 13e9, 70e9])  # 1B, 7B, 13B, 70B
tokens_optimal = params * 20                  # Chinchilla: ~20 tokens per param
tokens_overtrain = params * 100               # over-training for inference efficiency

print("Chinchilla-optimal training:")
print(f"{'Model Size':>12} {'Optimal Tokens':>16} {'Over-trained Tokens':>20}")
for p, t_opt, t_over in zip(params, tokens_optimal, tokens_overtrain):
    print(f"{p/1e9:>10.0f}B {t_opt/1e12:>14.1f}T {t_over/1e12:>18.1f}T")
print()
print("LLaMA-2 7B was trained on ~2T tokens (28x over Chinchilla optimal)")
print("→ Smaller model inference cost + more training = better per-$ at deployment")


## LLM Glossary (One-Line Definitions)

| Term | Definition |
|---|---|
| **Tokenization** | Mapping text to integer token IDs via BPE or similar |
| **Context window** | Maximum tokens the model can attend to at once |
| **KV cache** | Cached key-value matrices for past tokens to avoid recomputation |
| **Prefill** | Processing the input prompt (parallelizable, memory-bandwidth bound) |
| **Decode** | Generating output tokens one at a time (sequential, compute-bound) |
| **RLHF** | RL from Human Feedback — aligning model outputs with human preferences |
| **DPO** | Direct Preference Optimization — simpler alternative to full RLHF |
| **LoRA** | Low-Rank Adaptation — efficient fine-tuning via small rank-r matrices |
| **RAG** | Retrieval-Augmented Generation — fetch docs at query time, inject into context |
| **Hallucination** | Model generates plausible-sounding but factually incorrect text |
| **Grounding** | Linking model output to verifiable retrieved sources |
| **Speculative decoding** | Use small draft model to propose tokens; verify with large model |
| **GQA** | Grouped-Query Attention — fewer KV heads than Q heads; saves KV cache memory |
| **RoPE** | Rotary Position Embedding — positional encoding via rotation in embedding space |


## Common Interview Questions

**Q: When would you fine-tune vs use RAG vs just prompt-engineer?**
Prompt engineering first — it's zero cost and often sufficient. RAG when the task requires external or frequently-updated knowledge that doesn't fit in context. Fine-tune when you need to change the model's style, format, or learn a new skill (not just new facts), when prompting is too expensive at scale, or when you need strict output formatting that few-shot can't enforce reliably.

**Q: What are the practical tradeoffs of using an open model vs a closed API?**
Open models: full control, no data sharing, controllable latency, no per-token cost at scale — but require serving infra, updates, and engineering investment. Closed APIs: zero infra, state-of-the-art quality, simple scaling — but data privacy concerns, per-token costs at scale, vendor dependency, and rate limits. Most teams start with a closed API and migrate to open models if scale/privacy drives it.

**Q: What does "over-training" a model mean in the Chinchilla context?**
Chinchilla-optimal trains a model on ~20 tokens per parameter — the compute-optimal point. "Over-training" means using more tokens than Chinchilla-optimal for a given model size. This is done deliberately because inference efficiency matters: a smaller over-trained model (e.g., LLaMA-2 7B on 2T tokens) can match a larger undertrained model with lower serving cost per request.

**Q: What is the context window and why is it a constraint?**
The context window is the maximum number of tokens the model can attend to simultaneously. Attention is O(T²) in time and memory, so longer contexts are expensive. For RAG, the context window limits how many retrieved documents you can include. For agents, it limits conversation history depth. Modern models (GPT-4, Claude) support 128K–1M tokens but at higher cost.

## Key Takeaways
- Decoder-only dominates modern LLMs; encoder-decoder for structured seq2seq tasks
- Decision: prompt first → RAG if external knowledge → fine-tune if behavioral change needed
- Open vs closed: privacy/cost/control vs simplicity/quality/zero-infra
- Chinchilla: train ~20 tokens per param for compute-optimal; over-train for inference efficiency
- KV cache is the key memory optimization at inference; GQA reduces its memory footprint
- Fluency in the glossary (prefill, decode, KV cache, LoRA, RAG, RLHF) is table stakes for LLM roles